<a href="https://colab.research.google.com/github/Shineii86/InstaUserCheckBot/blob/main/notebooks/InstaUserCheckBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://capsule-render.vercel.app/api?type=waving&color=8B5CF6,06B6D4&height=200&section=header&text=Instagram%20Bot&fontSize=70&fontColor=ffffff&animation=fadeIn&fontAlignY=35&desc=Instagram%20Username%20Checker%20v3.0&descSize=20" alt="Instagram Bot">
  <h1>🔍 Instagram Username Checker</h1>
  <p><b>Check Instagram username availability at scale — CLI, Telegram Bot, or right here in Colab!</b></p>
</div>

---

### ✨ v3.0 Features
- 🧵 **Multi-threading** — Check dozens of usernames simultaneously
- 🌐 **Proxy Rotation** — Rotate HTTP/HTTPS/SOCKS proxies
- 🔄 **CSRF Session Reuse** — Auto-refreshing tokens with retry logic
- 🎲 **User-Agent Rotation** — Rotating 6-browser pool
- 🧠 **Word Combos** — Adjective + noun + number generation
- 🔮 **Pattern Templates** — Generate from `user_????`, `pro_####` patterns
- 📱 **Inline Query** — Check from any chat with `@botname username`
- 🌐 **Web App** — Full mini app inside Telegram
- 📊 **Session Stats** — Track hits, taken, rate limits, errors

---
## 📦 Step 1 — Install & Clone
Run this cell first to set up the environment.

In [ ]:
#@title 📦 Install & Clone Repository
#@markdown *Run this cell first — installs dependencies and clones the repo.*

import os, sys, subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ Error: {result.stderr.strip()}")
    return result

if not os.path.exists("InstaUserCheckBot"):
    print("📥 Cloning repository...")
    run("git clone https://github.com/Shineii86/InstaUserCheckBot.git")
else:
    print("✅ Repository already cloned.")

os.chdir("InstaUserCheckBot")
sys.path.insert(0, ".")

print("📦 Installing dependencies...")
run(f"{sys.executable} -m pip install -q python-telegram-bot requests")

from checker.config import Config
from checker.instagram import InstagramClient
from checker.generator import UsernameGenerator
print("\n✅ All dependencies ready!")

---
## ⚙️ Step 2 — Configure Settings
Fill in your credentials and adjust the checker settings.

In [ ]:
#@title ⚙️ Configuration
#@markdown ### 🔑 Credentials
TELEGRAM_BOT_TOKEN = ""  #@param {type:"string"}
TELEGRAM_CHAT_ID = ""    #@param {type:"string"}

#@markdown ---
#@markdown ### 🎯 Check Mode
MODE = "hits"  #@param ["hits", "count", "continuous"]
#@markdown *`hits` = stop after N available · `count` = check exactly N · `continuous` = run forever*
STOP_AFTER = 10  #@param {type:"slider", min:1, max:100, step:1}
MAX_ATTEMPTS = 100  #@param {type:"slider", min:10, max:1000, step:10}

#@markdown ---
#@markdown ### 🧬 Username Generation
USERNAME_LENGTH = 5  #@param {type:"slider", min:3, max:20, step:1}
GENERATION_MODE = "random"  #@param ["random", "word_combo", "mixed"]

#@markdown ---
#@markdown ### 🚀 Performance
MAX_WORKERS = 10  #@param {type:"slider", min:1, max:20, step:1}
DELAY = 0.5  #@param {type:"slider", min:0.1, max:5.0, step:0.1}

#@markdown ---
#@markdown ### 💾 Output
OUTPUT_FILE = "available_usernames.txt"  #@param {type:"string"}

print("✅ Configuration saved!")
print(f"  Mode: {MODE} | Workers: {MAX_WORKERS} | Delay: {DELAY}s")
print(f"  Length: {USERNAME_LENGTH} | Gen: {GENERATION_MODE}")

---
## 🚀 Step 3 — Run Username Checker (CLI Mode)
Start checking usernames based on your Step 2 configuration.

In [ ]:
#@title 🚀 Run Username Checker
#@markdown *Starts checking based on your Step 2 configuration.*

import os, sys, signal
sys.path.insert(0, ".")

from checker.config import Config
from checker.core import Checker

config = Config(
    telegram_token=TELEGRAM_BOT_TOKEN,
    telegram_chat_id=TELEGRAM_CHAT_ID,
    username_length=USERNAME_LENGTH,
    generation_mode=GENERATION_MODE,
    mode=MODE,
    stop_after_hits=STOP_AFTER,
    max_attempts=MAX_ATTEMPTS,
    max_workers=MAX_WORKERS,
    delay=DELAY,
    output_file=OUTPUT_FILE,
)

errors = config.validate()
if errors:
    for e in errors:
        print(f"❌ {e}")
else:
    print("🔍 Starting InstaUserCheckBot...")
    print("═" * 50)
    print(f"  Mode: {config.mode} | Workers: {config.max_workers} | Delay: {config.delay}s")
    print(f"  Length: {config.username_length} | Gen: {config.generation_mode}")
    print("═" * 50)
    
    checker = Checker(config)
    signal.signal(signal.SIGINT, lambda *_: checker.stop())
    
    stats = checker.run()
    print(f"\n🎉 Done! Found {stats.hits} available usernames.")

---
## 🤖 Step 4 — Launch Telegram Bot
Run the interactive Telegram bot directly from Colab.

In [ ]:
#@title 🤖 Launch Telegram Bot
#@markdown ---
#@markdown ### 🔑 Paste your Bot Token
#@markdown Get one from [@BotFather](https://t.me/BotFather) → `/newbot`
BOT_TOKEN_HERE = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### ▶️ Then run this cell and open your bot in Telegram!

import os, sys
from IPython.display import display, HTML
sys.path.insert(0, ".")

# Fix event loop for Jupyter/Colab
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nest_asyncio"])
    import nest_asyncio
    nest_asyncio.apply()

if not BOT_TOKEN_HERE:
    print("❌ Please enter your BOT_TOKEN above.")
    print("   Get one from: https://t.me/BotFather → /newbot")
else:
    os.environ["TELEGRAM_BOT_TOKEN"] = BOT_TOKEN_HERE

    display(HTML('''
    <div style="background: linear-gradient(135deg, #8B5CF6 0%, #06B6D4 100%); color: white;
                padding: 20px; border-radius: 12px; font-family: sans-serif; margin: 10px 0;">
        <h3 style="margin:0 0 10px 0;">🤖 Bot is Starting...</h3>
        <p style="margin:0; opacity:0.9; line-height: 1.8;">
            ✅ Open <b>Telegram</b> and find your bot<br>
            ✅ Send <code>/start</code> to begin<br>
            ✅ Try: <code>/check username</code> or just type a username<br>
            ✅ Try: <code>/pattern user_????</code> for pattern generation<br>
            ✅ Try: <code>@botname username</code> for inline queries<br>
            ✅ <b>Keep this cell running!</b> Stopping it kills the bot
        </p>
    </div>
    '''))

    print("\n" + "═" * 50)
    print("  🤖 InstaUserCheckBot is LIVE")
    print("═" * 50)
    print("  📱 Commands: /start /check /batch /generate /pattern /settings /stats /history /ping /about /stop")
    print("  💡 Quick:    Just type any username to check it")
    print("  🔍 Inline:   @botname username from any chat")
    print("  🛑 Stop:    Click ⏹️ on this cell")
    print("═" * 50)
    print()

    try:
        from bot.handlers import run_bot
        run_bot(BOT_TOKEN_HERE)
    except KeyboardInterrupt:
        print("\n🛑 Bot stopped.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("   Make sure your token is valid (get one from @BotFather).")

---
## 🔍 Step 5 — Quick Single Check
Check a single Instagram username without running the full bot.

In [ ]:
#@title 🔍 Check a Single Username

USERNAME_TO_CHECK = ""  #@param {type:"string"}
#@markdown *Enter a username without the @ symbol (e.g., `myusername`)*

import sys
sys.path.insert(0, ".")

from checker.instagram import InstagramClient, AVAILABLE, TAKEN, RATE_LIMITED, ERROR

if not USERNAME_TO_CHECK:
    print("⚠️ Please enter a username above.")
else:
    username = USERNAME_TO_CHECK.strip().lower().lstrip("@")
    
    if len(username) < 1 or len(username) > 30:
        print(f"🚫 @{username} — Instagram usernames are 1-30 chars.")
    elif ' ' in username:
        print(f"🚫 @{username} — No spaces allowed.")
    else:
        print(f"🔍 Checking @{username}...")
        
        client = InstagramClient([
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        ])
        
        status, _ = client.check(username)
        
        if status == AVAILABLE:
            print(f"\n🎉 @{username} is AVAILABLE!")
            print(f"   Claim it: https://instagram.com/{username}")
        elif status == TAKEN:
            print(f"\n❌ @{username} is taken.")
        elif status == RATE_LIMITED:
            print(f"\n⚠️ Rate limited. Try again later or use a proxy.")
        else:
            print(f"\n💥 Error checking @{username}.")

---
## 📋 Step 6 — Check Multiple Usernames
Paste multiple usernames and check them all at once.

In [ ]:
#@title 📋 Check Multiple Usernames

import sys, time
sys.path.insert(0, ".")

from checker.instagram import InstagramClient, AVAILABLE, TAKEN, RATE_LIMITED, ERROR

# Enter usernames below (one per line or comma-separated)
USERNAMES = """instagram
testuser123
coolname
myaccount99"""  #@param {type:"raw"}

DELAY_BETWEEN = 0.5  #@param {type:"slider", min:0.1, max:3.0, step:0.1}

# Parse usernames
names = []
for line in str(USERNAMES).split('\n'):
    for part in line.split(','):
        name = part.strip().lower().lstrip('@')
        if name and 1 <= len(name) <= 30:
            names.append(name)

if not names:
    print("⚠️ No valid usernames entered.")
else:
    print(f"🔍 Checking {len(names)} usernames...\n")
    
    client = InstagramClient([
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ])
    
    results = {AVAILABLE: [], TAKEN: [], RATE_LIMITED: [], ERROR: []}
    start = time.time()
    
    for i, name in enumerate(names):
        status, _ = client.check(name, delay=DELAY_BETWEEN)
        results[status].append(name)
        emoji = {AVAILABLE: '✅', TAKEN: '❌', RATE_LIMITED: '⚠️', ERROR: '💥'}[status]
        print(f"  {emoji} @{name} — {status}")
    
    elapsed = time.time() - start
    print(f"\n{'═' * 40}")
    print(f"  📊 Results:")
    print(f"  ✅ Available: {len(results[AVAILABLE])}")
    print(f"  ❌ Taken:     {len(results[TAKEN])}")
    print(f"  ⚠️ Rate Limit: {len(results[RATE_LIMITED])}")
    print(f"  💥 Errors:    {len(results[ERROR])}")
    print(f"  ⏱️ Time:      {elapsed:.1f}s")
    
    if results[AVAILABLE]:
        print(f"\n  🎉 Available usernames:")
        for name in results[AVAILABLE]:
            print(f"    ✅ @{name}")

---
## 🔮 Step 7 — Pattern Template Check
Generate usernames from a pattern and check them.

**Pattern Syntax:**
- `?` = random letter (a-z)
- `#` = random digit (0-9)
- `!` = letter or digit
- `_` = literal underscore
- Other characters are used literally

**Examples:** `user_????`, `pro_####`, `my_!_!_name`

In [ ]:
#@title 🔮 Pattern Template Check

PATTERN = "user_????"  #@param {type:"string"}
PATTERN_COUNT = 20  #@param {type:"slider", min:5, max:100, step:5}
PATTERN_DELAY = 0.5  #@param {type:"slider", min:0.1, max:3.0, step:0.1}

import sys, time
sys.path.insert(0, ".")

from checker.instagram import InstagramClient, AVAILABLE, TAKEN, RATE_LIMITED, ERROR
from checker.generator import UsernameGenerator

error = UsernameGenerator.validate_pattern(PATTERN)
if error:
    print(f"❌ {error}")
else:
    gen = UsernameGenerator()
    names = []
    seen = set()
    attempts = 0
    while len(names) < PATTERN_COUNT and attempts < PATTERN_COUNT * 20:
        attempts += 1
        name = ""
        for ch in PATTERN:
            if ch == '?': name += 'abcdefghijklmnopqrstuvwxyz'[__import__('random').randint(0,25)]
            elif ch == '#': name += '0123456789'[__import__('random').randint(0,9)]
            elif ch == '!': name += 'abcdefghijklmnopqrstuvwxyz0123456789'[__import__('random').randint(0,35)]
            else: name += ch
        if name not in seen and len(name) >= 1 and len(name) <= 30:
            seen.add(name)
            names.append(name)
    
    print(f"🔮 Pattern: {PATTERN} | Generated: {len(names)} names\n")
    
    client = InstagramClient([
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ])
    
    results = {AVAILABLE: [], TAKEN: [], RATE_LIMITED: [], ERROR: []}
    start = time.time()
    
    for name in names:
        status, _ = client.check(name, delay=PATTERN_DELAY)
        results[status].append(name)
        emoji = {AVAILABLE: '✅', TAKEN: '❌', RATE_LIMITED: '⚠️', ERROR: '💥'}[status]
        print(f"  {emoji} @{name}")
    
    elapsed = time.time() - start
    print(f"\n{'═' * 40}")
    print(f"  ✅ Available: {len(results[AVAILABLE])} | ❌ Taken: {len(results[TAKEN])}")
    print(f"  ⏱️ Time: {elapsed:.1f}s ({len(names)/elapsed:.1f}/sec)")
    
    if results[AVAILABLE]:
        print(f"\n  🎉 Available:")
        for name in results[AVAILABLE]:
            print(f"    ✅ @{name}")

---

## 📚 Resources

| | |
|---|---|
| 📖 **Repo** | [github.com/Shineii86/InstaUserCheckBot](https://github.com/Shineii86/InstaUserCheckBot) |
| 🤖 **BotFather** | [@BotFather](https://t.me/BotFather) — create your bot |
| 🆔 **Chat ID** | [@userinfobot](https://t.me/userinfobot) — get your ID |
| 📋 **Changelog** | [CHANGELOG.md](https://github.com/Shineii86/InstaUserCheckBot/blob/main/CHANGELOG.md) |

---

<div align="center">

**Made with ❤️ by [@Shineii86](https://github.com/Shineii86)**

⭐ Star the repo if it helped you find a great username!

</div>